In [ ]:
# Importa as bibliotecas necessárias
import pandas as pd
import os

# Mensagem inicial para indicar o início do processo
print("Iniciando a formatação final do Excel Multidimensional...")

# Define o caminho para a pasta do banco de dados
caminho_bd = '../../banco_de_dados/'
# Define o caminho do arquivo CSV de entrada
caminho_csv = caminho_bd + 'Base_Analitica_Multidimensional_Calculada.csv'
# Lê o arquivo CSV principal, garantindo que a coluna CD_SETOR seja tratada como texto
df_ivs = pd.read_csv(caminho_csv, sep=';', dtype={'CD_SETOR': str})

# 1. Cria o Dicionário de Dados Atualizado (Fase 2) como uma lista de dicionários
dados_dic = [
    {'Variável Final': 'CD_SETOR', 'Tipo': 'Texto', 'Descrição': 'Código oficial do Setor Censitário (IBGE).'},
    {'Variável Final': 'Moradia_Predominante', 'Tipo': 'Texto', 'Descrição': 'Morfologia Urbana (Casa, Apartamento, Cortiço, etc).'},
    {'Variável Final': 'DENOM_Total_Lares', 'Tipo': 'Numérico', 'Descrição': 'Universo Base (V01042 - Total de Pessoas Responsáveis). Substitui os domicílios físicos para evitar coabitação oculta.'},
    {'Variável Final': 'DENOM_Pop_Total', 'Tipo': 'Numérico', 'Descrição': 'Universo Demográfico Total do Setor.'},
    {'Variável Final': 'DENOM_Pop_15_Mais', 'Tipo': 'Numérico', 'Descrição': 'Universo Base para cálculos de Educação/Alfabetização.'},
    {'Variável Final': 'ind_agua_inadequada', 'Tipo': 'Numérico (0 a 1)', 'Descrição': 'Proporção de lares sem acesso a rede geral de água.'},
    {'Variável Final': 'ind_esgoto_inadequado', 'Tipo': 'Numérico (0 a 1)', 'Descrição': 'Proporção de lares com esgotamento sanitário precário ou inexistente.'},
    {'Variável Final': 'ind_lixo_inadequado', 'Tipo': 'Numérico (0 a 1)', 'Descrição': 'Proporção de lares com destinação de lixo inadequada.'},
    {'Variável Final': 'ind_analfabetismo', 'Tipo': 'Numérico (0 a 1)', 'Descrição': 'Proporção de adultos analfabetos.'},
    {'Variável Final': 'ind_cor_raca', 'Tipo': 'Numérico (0 a 1)', 'Descrição': 'Proporção de autodeclarados pretos, pardos ou indígenas.'},
    {'Variável Final': 'ind_densidade_habitacional', 'Tipo': 'Numérico (0 a 1)', 'Descrição': 'Normalização Min-Max da Razão de moradores por responsável.'},
    {'Variável Final': 'ind_pobreza_multidimensional', 'Tipo': 'Numérico (0 a 1)', 'Descrição': 'Proxy de Extrema Pobreza combinando Renda Invertida (40%), Moradias Improvisadas (20%), Ausência de Banheiro (20%) e Dependência Jovem (20%).'}
]
# Converte a lista de dicionários em um DataFrame
df_dic_dados = pd.DataFrame(dados_dic)

# 2. Cria o Dicionário de Aplicação Atualizado (Fase 2) como uma lista de dicionários
dados_app = [
    {'Dimensão': 'Saneamento Básico', 'Fórmula Aplicada': '(Riscos Específicos) / Total de Lares (Responsáveis)', 'Variáveis Originais (Censo 2022)': 'Somatório de Riscos / V01042'},
    {'Dimensão': 'Educação', 'Fórmula Aplicada': 'Analfabetos (15 anos ou mais) / População (15 anos ou mais)', 'Variáveis Originais (Censo 2022)': 'V00901 / V00900'},
    {'Dimensão': 'Demografia (Cor/Raça)', 'Fórmula Aplicada': '(População Preta + Parda + Indígena) / População Total', 'Variáveis Originais (Censo 2022)': '(V01318 + V01320 + V01321) / v0001'},
    {'Dimensão': 'Habitação (Densidade Manual)', 'Fórmula Aplicada': '(População em Casas + Pop. em Tendas) / Total de Lares', 'Variáveis Originais (Censo 2022)': '(V00005 + V00006) / V01042'},
    {'Dimensão': 'Extrema Pobreza Multidimensional', 'Fórmula Aplicada': '(Renda_Inv * 0.4) + (Sem_Banho * 0.2) + (Sem_Abrigo * 0.2) + (Crianças * 0.2)', 'Variáveis Originais (Censo 2022)': 'V06004, V00236, V00238, V00002, V01031 a 1033'}
]
# Converte a lista de dicionários em um DataFrame
df_dic_app = pd.DataFrame(dados_app)

# 3. Gera a Prova Real Estatística (descrição dos indicadores calculados)
colunas_calculadas = ['ind_agua_inadequada', 'ind_esgoto_inadequado', 'ind_lixo_inadequado', 
                      'ind_analfabetismo', 'ind_cor_raca', 'ind_densidade_habitacional', 'ind_pobreza_multidimensional']
# Calcula estatísticas descritivas para os indicadores selecionados
df_prova = df_ivs[colunas_calculadas].describe().T[['min', 'max', 'mean']]
# Conta o número de valores nulos em cada indicador e adiciona ao DataFrame de estatísticas
df_prova['valores_nulos'] = df_ivs[colunas_calculadas].isnull().sum()
# Ajusta o índice e renomeia as colunas para melhor apresentação
df_prova = df_prova.reset_index().rename(columns={'index': 'Indicador Matemático', 'min': 'Valor Mínimo', 'max': 'Valor Máximo', 'mean': 'Média Geral'})

# 4. Exporta todos os dados para um arquivo Excel com múltiplas abas e formatação visual
caminho_excel_final = caminho_bd + 'Base_IVS_Multidimensional_Formatada.xlsx'
# Mensagem para indicar o início da exportação
print("Gerando o arquivo Excel com 4 abas... (Aguarde, processando 450 mil linhas)")

# Cria o arquivo Excel e adiciona as abas
with pd.ExcelWriter(caminho_excel_final, engine='xlsxwriter') as writer:
    # Exporta a base analítica principal para a primeira aba
    df_ivs.to_excel(writer, sheet_name='Base_Analitica', index=False)
    # Exporta o dicionário de dados para a segunda aba
    df_dic_dados.to_excel(writer, sheet_name='Dicionario_Dados', index=False)
    # Exporta o dicionário de aplicação/metodologia para a terceira aba
    df_dic_app.to_excel(writer, sheet_name='Metodologia_Formulas', index=False)
    # Exporta as estatísticas descritivas para a quarta aba
    df_prova.to_excel(writer, sheet_name='Prova_Real_Estatistica', index=False)
    
    # Formatação Visual das abas do Excel
    workbook = writer.book
    # Define o formato do cabeçalho (negrito, cor de fundo, borda, alinhamento)
    formato_cabecalho = workbook.add_format({'bold': True, 'bg_color': '#1F497D', 'font_color': 'white', 'border': 1, 'align': 'center', 'valign': 'vcenter'})
    # Define o formato para números (4 casas decimais, borda)
    formato_numeros = workbook.add_format({'num_format': '0.0000', 'border': 1})
    # Define o formato para texto (borda, quebra de linha, alinhamento superior)
    formato_texto = workbook.add_format({'border': 1, 'text_wrap': True, 'valign': 'top'})
    
    # Formatação da primeira aba (Base Analítica)
    ws1 = writer.sheets['Base_Analitica']
    # Aplica o formato de cabeçalho para cada coluna
    for col_num, value in enumerate(df_ivs.columns.values):
        ws1.write(0, col_num, value, formato_cabecalho)
    # Define largura e formato das colunas de acordo com o tipo de dado
    ws1.set_column('A:A', 18, formato_texto) 
    ws1.set_column('B:E', 22, formato_texto) 
    ws1.set_column('F:I', 15, formato_numeros) # Denominadores (Inteiros)
    ws1.set_column('J:P', 20, formato_numeros) # Índices (0 a 1)
    # Congela o cabeçalho para facilitar navegação
    ws1.freeze_panes(1, 0)
    # Adiciona filtro automático para todas as colunas
    ws1.autofilter(0, 0, len(df_ivs), len(df_ivs.columns) - 1)
    
    # Formatação das abas 2, 3 e 4 (Dicionário, Metodologia, Prova)
    for aba, df_temp in zip(['Dicionario_Dados', 'Metodologia_Formulas', 'Prova_Real_Estatistica'], [df_dic_dados, df_dic_app, df_prova]):
        ws = writer.sheets[aba]
        # Aplica o formato de cabeçalho para cada coluna
        for col_num, value in enumerate(df_temp.columns.values):
            ws.write(0, col_num, value, formato_cabecalho)
        # Define largura das colunas para melhor visualização
        ws.set_column('A:D', 35, formato_texto)

# Mensagem final indicando sucesso e o caminho do arquivo gerado
print(f"SUCESSO! O Excel definitivo para a reunião foi salvo em:\n{caminho_excel_final}")

Iniciando a formatação final do Excel Multidimensional...
Gerando o arquivo Excel com 4 abas... (Aguarde, processando 450 mil linhas)
SUCESSO! O Excel definitivo para a reunião foi salvo em:
../../banco_de_dados/Base_IVS_Multidimensional_Formatada.xlsx
